In [37]:
import warnings
warnings.filterwarnings('ignore')

import os
import sys
import json
import time

import numpy as np
import networkx as nx
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# import fleet_opt_core

import src
import figures

# from src.rust_bridge import build_rust_network
from tqdm import tqdm

from src.reload import deep_reload
from figures import style
matplotlib.rcParams.update(**style['rcparams'])

## Constrained optimization example

In [25]:
'''
Loading graphs

trips - a graph with trips as nodes and edges to possible successors
locations - a graph representing the physical cost of traveling between
    trip terminals and depots
'''
deep_reload(src)

trips = src.graph.graph_from_json('Data/Unitrans/trips.json')

trips.remove_nodes_from(
    [k for k, n in trips._node.items() if \
     not n['route'] in ['A', 'B', 'C', 'D', 'E', 'F']]
)

locations = src.graph.graph_from_json('Data/Unitrans/locations.json')

locations.remove_node('Secondary Depot')

In [61]:
'''
Adding placeholder nodes
'''

data = {
    'start_location': 'Primary Depot',
    'finish_location': 'Primary Depot',
    'route': 'Depot',
    'distance': 0.,
    'duration': 0,
    'start': 0,
    'finish': 0,
}

trips.add_edges_from([('depot_s', t) for t in trips.nodes])
trips.add_edges_from([(s, 'depot_t') for s in trips.nodes])

trips.add_node('depot_s', **data)
trips.add_node('depot_t', **data)

for s, t, e in trips.edges(data=True):

    sn, tn = trips._node[s]['finish_location'], trips._node[t]['start_location']

    e['distance'] = locations._adj[sn][tn]['distance']
    e['duration'] = locations._adj[sn][tn]['duration']

In [62]:
'''
All possible depot-depot paths
'''
from itertools import islice, pairwise

asp = nx.shortest_simple_paths(trips, 'depot_s', 'depot_t', weight='distance')

path_distance = lambda g, p: sum(
    [g[s][t]['distance'] + g._node[t]['distance'] for s, t in pairwise(p)]
)
path_duration = lambda g, p: sum(
    [g[s][t]['duration'] + g._node[t]['duration'] for s, t in pairwise(p)]
)

paths = []
distances = []
durations = []

for i in tqdm(range(1000)):

    p = next(asp)
    pdi = path_distance(trips, p)
    pdu = path_duration(trips, p)

    if (pdi >= 300000) or (pdu >= 4 * 3600):

        break

    paths.append(p)
    distances.append(pdi)
    durations.append(pdu)

 48%|████▊     | 477/1000 [00:03<00:04, 130.57it/s]


KeyboardInterrupt: 

In [ ]:
paths[-1], distances[-1] / 1e3, durations[-1] / 3.6e3

In [60]:
n = np.unique([n for p in paths for n in p])
n, n.shape

(array(['C_31_inbound_0750', 'C_31_inbound_0815', 'C_31_inbound_0850',
        'C_31_inbound_0950', 'C_31_inbound_1020', 'C_31_inbound_1050',
        'C_31_inbound_1120', 'C_31_inbound_1150', 'C_31_inbound_1300',
        'C_31_inbound_1330', 'C_31_inbound_1400', 'C_31_inbound_1430',
        'C_31_inbound_1500', 'C_31_inbound_1530', 'C_31_inbound_1600',
        'C_31_inbound_1700', 'C_31_inbound_1800', 'C_31_outbound_0655',
        'C_31_outbound_0725', 'C_31_outbound_0755', 'C_31_outbound_0900',
        'C_31_outbound_0930', 'C_31_outbound_1000', 'C_31_outbound_1030',
        'C_31_outbound_1100', 'C_31_outbound_1210', 'C_31_outbound_1240',
        'C_31_outbound_1310', 'C_31_outbound_1340', 'C_31_outbound_1410',
        'C_31_outbound_1440', 'C_31_outbound_1510', 'C_31_outbound_1610',
        'C_31_outbound_1710', 'D_31_inbound_0850', 'D_31_inbound_0950',
        'D_31_inbound_1050', 'D_31_inbound_1150', 'D_31_inbound_1300',
        'D_31_inbound_1400', 'D_31_inbound_1500', 'D_31_inbo

In [59]:
trips.number_of_nodes(), list(trips.nodes)

(324,
 ['A_31_outbound_0655',
  'A_31_outbound_0755',
  'A_31_outbound_0900',
  'A_31_outbound_1000',
  'A_31_outbound_1100',
  'A_31_outbound_1210',
  'A_31_outbound_1310',
  'A_31_outbound_1410',
  'A_31_outbound_1510',
  'A_31_outbound_1610',
  'A_31_outbound_1710',
  'A_31_outbound_1810',
  'A_31_outbound_1910',
  'A_31_outbound_2010',
  'A_31_outbound_2040',
  'A_31_outbound_2140',
  'A_31_inbound_0750',
  'A_31_inbound_0850',
  'A_31_inbound_0950',
  'A_31_inbound_1050',
  'A_31_inbound_1150',
  'A_31_inbound_1300',
  'A_31_inbound_1400',
  'A_31_inbound_1500',
  'A_31_inbound_1600',
  'A_31_inbound_1700',
  'A_31_inbound_1800',
  'A_31_inbound_1900',
  'A_31_inbound_2000',
  'A_31_inbound_2100',
  'A_31_inbound_2130',
  'A_31_inbound_2230',
  'B_31_outbound_0655',
  'B_31_outbound_0725',
  'B_31_outbound_0755',
  'B_31_outbound_0825',
  'B_31_outbound_0900',
  'B_31_outbound_0930',
  'B_31_outbound_1000',
  'B_31_outbound_1030',
  'B_31_outbound_1100',
  'B_31_outbound_1130',
  

In [54]:
[trips[s][t]['distance'] + trips._node[t]['distance'] for s, t in pairwise(paths[-1])]

[3685.606929159698,
 2505.228171834096,
 2830.8861827280184,
 2505.228171834096,
 2830.8861827280184,
 2505.228171834096,
 2830.8861827280184,
 2505.228171834096,
 2830.8861827280184,
 2505.228171834096,
 2830.8861827280184,
 2505.228171834096,
 2830.8861827280184,
 2505.228171834096,
 2830.8861827280184,
 2505.228171834096,
 2830.8861827280184,
 2505.228171834096,
 2830.8861827280184,
 2505.228171834096,
 4252.088717931519,
 6210.290827579405,
 885.48602817697]